# VIX Multi-Horizon Ensemble — Stacking sur 28 modèles de référence
## Horizons : 1j, 2j, 3j, 5j, 7j, 10j | Régimes : CALM / NORMAL / STRESS / GLOBAL
## Méta-modèle XGBoost sur les probabilités OOF de tous les modèles


In [ ]:
import sys
!{sys.executable} -m pip install -q xgboost lightgbm yfinance pandas_datareader arch pykalman hmmlearn shap xlsxwriter imbalanced-learn statsmodels

import os, time, json, warnings, random
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader.data as web
import shap

from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from imblearn.combine import SMOTETomek
from arch import arch_model
from pykalman import KalmanFilter
from hmmlearn import hmm as hmmlib

SEED = 42
random.seed(SEED); np.random.seed(SEED)
print("Imports OK")


In [ ]:
# =============================================================================
# CONFIGURATION CENTRALE
# =============================================================================
CONFIG = {
    'seed':           42,
    'start_date':     '2000-01-01',
    'flat_thr':       0.003,
    'class_quantiles': [0.25, 0.75],
    'class_labels':   ['DOWN_FORT','DOWN_FAIBLE','UP_FAIBLE','UP_FORT'],
    'test_date':      None,   # calculé dynamiquement (80/20 chronologique)
    # Stacking
    'meta_n_estimators': 300,
    'meta_max_depth':    4,
    'meta_lr':           0.05,
    'n_folds_oof':       5,
    # Horizons à entraîner
    'horizons':       [1, 2, 3, 5, 7, 10],
    # Régimes
    'regimes':        ['CALM','NORMAL','STRESS','GLOBAL'],
}

TARGET_COL = 'VIX_Amplitude_Class'

YF_TICKERS = """
^GSPC ^IXIC ^VIX ^VXN ^OVX ^GVZ ^EVZ ^VVIX
^FTSE ^N225 ^HSI ^GDAXI ^STOXX50E
SPY QQQ TLT GLD USO HYG LQD
AAPL AMZN MSFT NVDA INTC QCOM XOM WMT MCD SBUX
MS COF BLK SCHW CLX CPB LMT NOC GD HON
CCI PSA EQIX NEE TXN PAYX LUV CMCSA
XLK XLF XLE XLV XLU XLB XLI XLY
""".split()

FRED_SERIES = {'NFCI':'NFCI','STLFSI':'STLFSI4','T10Y2Y':'T10Y2Y','EFFR':'EFFR'}

# =============================================================================
# MODÈLES DE RÉFÉRENCE — extraits de tous les runs précédents
# F1_dir >= 0.50, leakage Kalman filtré, dédupliqués par (algo, regime, horizon)
# =============================================================================
REFERENCE_MODELS = [
  {
    "model_id": "egarch_v2_7j_CALM_LightGBM",
    "source": "egarch_v2",
    "algo": "LightGBM",
    "regime": "CALM",
    "horizon": 7,
    "F1_dir": 0.6396,
    "F1_UP_FORT": 0.3299,
    "F1_DOWN_FORT": 0.3902,
    "n_features": 6,
    "train_start": "2001-02-19",
    "sampler": "SMOTE",
    "best_params": "{}",
    "features": [
      "NFCI_ret_5d__div__NVDA_vol_20d",
      "QCOM_ret_20d__zrel__PCE_zscore_60d",
      "vix_vs_ma20__zrel__XLK_Tech_zscore_60d",
      "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d",
      "VIX_Price_zscore_60d__div__PCE_zscore_60d",
      "DAX_Germany_zscore_60d"
    ]
  },
  {
    "model_id": "egarch_v1_7j_GLOBAL_LightGBM",
    "source": "egarch_v1",
    "algo": "LightGBM",
    "regime": "GLOBAL",
    "horizon": 7,
    "F1_dir": 0.6348,
    "F1_UP_FORT": 0.4307,
    "F1_DOWN_FORT": 0.4733,
    "n_features": 17,
    "train_start": "2001-02-07",
    "sampler": "SMOTE",
    "best_params": "{}",
    "features": [
      "NFCI_ret_5d__div__VRP",
      "NFCI_ret_5d__ret5x__NFCI_ret_20d",
      "CPB_CampbellSoup_vol_20d",
      "CLX_Clorox_vol_20d",
      "US3Y_Rate_ret_5d",
      "EWA_Australia_zscore_60d",
      "XLV_Health_zscore_60d",
      "CTAS_Cintas_vol_20d",
      "GILD_Gilead_ret_20d",
      "heston_xi__prod__kalman_filtered",
      "EWS_Singapore_ret_5d",
      "VVIX_vol_20d__zrel__kalman_filtered",
      "Nikkei_Japan_vol_20d",
      "SBUX_ret_5d",
      "VVIX_vol_20d__prod__kalman_filtered",
      "PAYX_Paychex_ret_20d",
      "NFCI_ret_5d__div__NFCI_vol_20d"
    ]
  },
  {
    "model_id": "egarch_v2_7j_GLOBAL_RandomForest",
    "source": "egarch_v2",
    "algo": "RandomForest",
    "regime": "GLOBAL",
    "horizon": 7,
    "F1_dir": 0.6307,
    "F1_UP_FORT": 0.3951,
    "F1_DOWN_FORT": 0.4944,
    "n_features": 16,
    "train_start": "2001-02-06",
    "sampler": "SMOTE",
    "best_params": "{}",
    "features": [
      "VIX_Price_zscore_60d__div__NFCI_vol_20d",
      "Core_CPI_zscore_60d",
      "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d",
      "Nikkei_Japan_zscore_60d",
      "XLK_Tech_zscore_60d",
      "NOC_Northrop_ret_20d",
      "CLX_Clorox_vol_20d",
      "vix_zscore_10d__div__heston_var_ev_h7",
      "Nikkei_Japan_vol_20d",
      "vix_vol_of_vol_10d__minus__NFCI_vol_20d",
      "VRP__macross__hmm_p_stress",
      "HD_zscore_60d",
      "Industrial_Production_zscore_60d",
      "NFCI_ret_5d__div__NFCI_vol_20d",
      "ASX_Australia_vol_20d",
      "Michigan_Sentiment_ret_20d"
    ]
  },
  {
    "model_id": "egarch_v2_7j_NORMAL_RandomForest",
    "source": "egarch_v2",
    "algo": "RandomForest",
    "regime": "NORMAL",
    "horizon": 7,
    "F1_dir": 0.6292,
    "F1_UP_FORT": 0.489,
    "F1_DOWN_FORT": 0.4629,
    "n_features": 18,
    "train_start": "2001-02-06",
    "sampler": "SMOTEENN",
    "best_params": "{}",
    "features": [
      "NFCI_ret_5d__minus__NFCI_vol_20d",
      "VIX_Price_zscore_60d__div__heston_var_ev_h1",
      "VIX_Price_zscore_60d__div__NFCI_vol_20d",
      "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d",
      "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d",
      "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d",
      "HD_ret_20d",
      "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d",
      "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d",
      "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d",
      "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d",
      "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d",
      "Nikkei_Japan_vol_20d",
      "NFCI_ret_5d__div__NFCI_vol_20d",
      "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
      "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d",
      "vix_vol_of_vol_10d__prod__vix_level",
      "vix_zscore_10d__zrel__EWP_Spain_zscore_60d"
    ]
  },
  {
    "model_id": "egarch_v2_1j_CALM_LightGBM",
    "source": "egarch_v2",
    "algo": "LightGBM",
    "regime": "CALM",
    "horizon": 1,
    "F1_dir": 0.6277,
    "F1_UP_FORT": 0.1519,
    "F1_DOWN_FORT": 0.34,
    "n_features": 7,
    "train_start": "2001-08-01",
    "sampler": "SMOTE",
    "best_params": "{}",
    "features": [
      "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
      "hmm_p_stress__minus__spx_vol_5d",
      "XOM_ret_1d",
      "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
      "MS_MorganStanley_ret_5d",
      "VIX_Price_zscore_60d__div__VRP",
      "MSTR_Bitcoin3_ret_5d"
    ]
  },
  {
    "model_id": "egarch_v1_5j_CALM_LightGBM",
    "source": "egarch_v1",
    "algo": "LightGBM",
    "regime": "CALM",
    "horizon": 5,
    "F1_dir": 0.6232,
    "F1_UP_FORT": 0.1556,
    "F1_DOWN_FORT": 0.4748,
    "n_features": 16,
    "train_start": "2000-11-16",
    "sampler": "SMOTETomek",
    "best_params": "{}",
    "features": [
      "XLY_Disc_vol_20d",
      "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d",
      "NFCI_ret_5d",
      "EWA_Australia_zscore_60d",
      "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d",
      "MSTR_Bitcoin3_ret_1d",
      "vix_vs_ma10__div__vix_zscore_10d",
      "AMT_AmericanTower_ret_1d",
      "Brent_Oil_FRED_ret_5d",
      "vix_zscore_10d__zrel__AMZN_zscore_60d",
      "vix_zscore_10d__div__spx_abs_ret_max_5d",
      "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d",
      "CPB_CampbellSoup_zscore_60d",
      "SLB_Schlumberger_ret_1d",
      "HD_ret_1d",
      "vix_vs_ma10__minus__CAT_Caterpillar_ret_5d"
    ]
  },
  {
    "model_id": "egarch_v1_5j_GLOBAL_XGBoost",
    "source": "egarch_v1",
    "algo": "XGBoost",
    "regime": "GLOBAL",
    "horizon": 5,
    "F1_dir": 0.6195,
    "F1_UP_FORT": 0.4124,
    "F1_DOWN_FORT": 0.4619,
    "n_features": 8,
    "train_start": "2001-02-07",
    "sampler": "SMOTEENN",
    "best_params": "{}",
    "features": [
      "NFCI_ret_5d__div__vix_vol_of_vol_10d",
      "SJM_JM_Smucker_ret_5d",
      "NFCI_ret_5d__minus__vix_level",
      "vix_vol_of_vol_10d__prod__NFCI_zscore_60d",
      "NFCI_vol_20d__ret5x__HD_zscore_60d",
      "NFCI_ret_5d__div__NFCI_vol_20d",
      "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d",
      "NFCI_zscore_60d__minus__vix_ma_20"
    ]
  },
  {
    "model_id": "egarch_v2_7j_STRESS_LightGBM",
    "source": "egarch_v2",
    "algo": "LightGBM",
    "regime": "STRESS",
    "horizon": 7,
    "F1_dir": 0.6185,
    "F1_UP_FORT": 0.3265,
    "F1_DOWN_FORT": 0.4125,
    "n_features": 8,
    "train_start": "2000-11-15",
    "sampler": "BorderlineSMOTE",
    "best_params": "{}",
    "features": [
      "US3M_Rate_vol_20d",
      "AMD_zscore_60d__ret5x__XLU_Util_ret_5d",
      "DE_Deere_ret_5d",
      "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
      "VIX_Price_zscore_60d__prod__JNJ_ret_20d",
      "JNJ_ret_20d__macross__vix_vs_ma5",
      "US30Y_Rate_ret_20d",
      "VRP_ma5"
    ]
  },
  {
    "model_id": "egarch_v1_7j_CALM_RandomForest",
    "source": "egarch_v1",
    "algo": "RandomForest",
    "regime": "CALM",
    "horizon": 7,
    "F1_dir": 0.6129,
    "F1_UP_FORT": 0.2472,
    "F1_DOWN_FORT": 0.3684,
    "n_features": 5,
    "train_start": "2000-11-16",
    "sampler": "ADASYN",
    "best_params": "{}",
    "features": [
      "NFCI_ret_5d__div__NVDA_vol_20d",
      "NVDA_vol_20d__prod__ADBE_vol_20d",
      "US3Y_Rate_ret_5d",
      "XLF_Fin_vol_20d",
      "Retail_Sales_zscore_60d"
    ]
  },
  {
    "model_id": "egarch_v2_3j_NORMAL_GradientBoosting",
    "source": "egarch_v2",
    "algo": "GradientBoosting",
    "regime": "NORMAL",
    "horizon": 3,
    "F1_dir": 0.6123,
    "F1_UP_FORT": 0.3686,
    "F1_DOWN_FORT": 0.4161,
    "n_features": 17,
    "train_start": "2001-02-20",
    "sampler": "SMOTETomek",
    "best_params": "{}",
    "features": [
      "VIX_Price_zscore_60d__div__hmm_p_stress",
      "vix_zscore_10d__div__heston_var_ev_h1",
      "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
      "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
      "HangSeng_HK_ret_5d",
      "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
      "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
      "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
      "EWL_Switzerland_vol_20d",
      "EWM_Malaysia_vol_20d",
      "EOG_EOGResources_vol_20d",
      "kalman_filtered__prod__VOD_Vodafone_vol_20d",
      "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
      "GD_GeneralDynamics_zscore_60d",
      "ENB_EnbridgeInc_ret_1d",
      "INTC_ret_5d",
      "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d"
    ]
  },
  {
    "model_id": "egarch_v1_3j_CALM_LightGBM",
    "source": "egarch_v1",
    "algo": "LightGBM",
    "regime": "CALM",
    "horizon": 3,
    "F1_dir": 0.6111,
    "F1_UP_FORT": 0.233,
    "F1_DOWN_FORT": 0.3617,
    "n_features": 10,
    "train_start": "2000-11-03",
    "sampler": "BorderlineSMOTE",
    "best_params": "{}",
    "features": [
      "NFCI_ret_5d",
      "spx_abs_ret_max_5d",
      "vix_zscore_10d__div__XLY_Disc_vol_20d",
      "XLY_Disc_vol_20d",
      "NFCI_ret_5d__zrel__US3M_Rate_ret_5d",
      "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d",
      "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d",
      "vix_zscore_10d__minus__XLB_Materials_zscore_60d",
      "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d",
      "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d"
    ]
  },
  {
    "model_id": "egarch_v2_5j_GLOBAL_LogisticRegression",
    "source": "egarch_v2",
    "algo": "LogisticRegression",
    "regime": "GLOBAL",
    "horizon": 5,
    "F1_dir": 0.604,
    "F1_UP_FORT": 0.3119,
    "F1_DOWN_FORT": 0.4545,
    "n_features": 5,
    "train_start": "2001-02-06",
    "sampler": "BorderlineSMOTE",
    "best_params": "{}",
    "features": [
      "HON_Honeywell_zscore_60d__zrel__vix_vs_ma20",
      "VIX_Price_zscore_60d__div__NFCI_vol_20d",
      "vix_vs_ma10__zrel__US30Y_Rate_ret_20d",
      "PAYX_Paychex_vol_20d",
      "heston_var_ev_h7"
    ]
  },
  {
    "model_id": "egarch_v1_3j_GLOBAL_RandomForest",
    "source": "egarch_v1",
    "algo": "RandomForest",
    "regime": "GLOBAL",
    "horizon": 3,
    "F1_dir": 0.596,
    "F1_UP_FORT": 0.2843,
    "F1_DOWN_FORT": 0.427,
    "n_features": 15,
    "train_start": "2001-02-07",
    "sampler": "SMOTE",
    "best_params": "{}",
    "features": [
      "TXN_vol_20d",
      "NFCI_ret_5d__div__NFCI_vol_20d",
      "gjr_condvar_h1",
      "NFCI_ret_5d",
      "vix_vs_ma20__zrel__XLB_Materials_zscore_60d",
      "vix_zscore_10d__zrel__XLB_Materials_zscore_60d",
      "NWL_Newell_ret_20d__macross__VIX_zscore_60d",
      "heston_var_ev_h5",
      "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d",
      "Russell_Price_ret_5d__ret5x__HD_zscore_60d",
      "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d",
      "CCI_CrownCastle_vol_20d__macross__heston_kappa",
      "vix_vs_ma20__div__NWL_Newell_ret_20d",
      "NFCI_vol_20d__ret5x__heston_kappa",
      "NFCI_ret_5d__zrel__Russell_Price_ret_5d"
    ]
  },
  {
    "model_id": "egarch_v2_5j_CALM_XGBoost",
    "source": "egarch_v2",
    "algo": "XGBoost",
    "regime": "CALM",
    "horizon": 5,
    "F1_dir": 0.5904,
    "F1_UP_FORT": 0.1765,
    "F1_DOWN_FORT": 0.3299,
    "n_features": 17,
    "train_start": "2000-11-15",
    "sampler": "SMOTE",
    "best_params": "{}",
    "features": [
      "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
      "NFCI_ret_5d__div__NVDA_vol_20d",
      "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
      "VIX_Price_zscore_60d__div__VRP",
      "vix_zscore_10d__div__spx_abs_ret_max_5d",
      "EWY_Korea_ret_20d",
      "EWQ_France_zscore_60d",
      "LLY_zscore_60d",
      "EWC_Canada_zscore_60d",
      "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
      "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
      "XLY_Disc_vol_20d",
      "ASML_ASML_ret_5d__div__spx_momentum_3d",
      "NFCI_ret_5d",
      "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d",
      "T10Y2Y_Spread_ret_5d",
      "TM_Telephone_vol_20d"
    ]
  },
  {
    "model_id": "egarch_v1_5j_STRESS_XGBoost",
    "source": "egarch_v1",
    "algo": "XGBoost",
    "regime": "STRESS",
    "horizon": 5,
    "F1_dir": 0.5871,
    "F1_UP_FORT": 0.3768,
    "F1_DOWN_FORT": 0.5182,
    "n_features": 8,
    "train_start": "2000-11-16",
    "sampler": "BorderlineSMOTE",
    "best_params": "{}",
    "features": [
      "DE_Deere_vol_20d",
      "VIX_Price_zscore_60d__prod__PCE_zscore_60d",
      "Brent_Oil_FRED_ret_20d",
      "AMD_ret_5d",
      "SBUX_ret_5d",
      "NFCI_zscore_60d__minus__VIX_Price_zscore_60d",
      "LUV_SouthwestAir_ret_5d",
      "NFCI_zscore_60d__div__vix_mean_abs_ret_5d"
    ]
  },
  {
    "model_id": "egarch_v1_1j_STRESS_XGBoost",
    "source": "egarch_v1",
    "algo": "XGBoost",
    "regime": "STRESS",
    "horizon": 1,
    "F1_dir": 0.5839,
    "F1_UP_FORT": 0.321,
    "F1_DOWN_FORT": 0.466,
    "n_features": 8,
    "train_start": "2001-02-08",
    "sampler": "BorderlineSMOTE",
    "best_params": "{}",
    "features": [
      "ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d",
      "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d",
      "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d",
      "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d",
      "EXC_Exelon_zscore_60d",
      "vix_mean_abs_ret_5d",
      "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d",
      "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d"
    ]
  },
  {
    "model_id": "egarch_v1_1j_NORMAL_LightGBM",
    "source": "egarch_v1",
    "algo": "LightGBM",
    "regime": "NORMAL",
    "horizon": 1,
    "F1_dir": 0.5793,
    "F1_UP_FORT": 0.3231,
    "F1_DOWN_FORT": 0.3333,
    "n_features": 17,
    "train_start": "2000-11-16",
    "sampler": "SMOTETomek",
    "best_params": "{}",
    "features": [
      "heston_xi__minus__COF_CapitalOne_ret_5d",
      "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d",
      "US1Y_Rate_ret_20d",
      "heston_xi__minus__EQIX_Equinix_vol_20d",
      "VRP",
      "heston_xi__div__ASML_ASML_vol_20d",
      "vix_zscore_10d",
      "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d",
      "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d",
      "ROST_RossStores_ret_1d__macross__WMT_ret_20d",
      "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d",
      "NFCI_ret_5d__minus__ORCL_ret_1d",
      "MO_AltriaMG_ret_1d",
      "NFCI_ret_5d__zrel__ORCL_ret_1d",
      "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d",
      "heston_xi__minus__ADBE_vol_20d",
      "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d"
    ]
  },
  {
    "model_id": "egarch_v2_3j_CALM_LogisticRegression",
    "source": "egarch_v2",
    "algo": "LogisticRegression",
    "regime": "CALM",
    "horizon": 3,
    "F1_dir": 0.579,
    "F1_UP_FORT": 0.1818,
    "F1_DOWN_FORT": 0.3419,
    "n_features": 20,
    "train_start": "2000-11-02",
    "sampler": "SMOTE",
    "best_params": "{}",
    "features": [
      "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
      "NFCI_ret_5d__prod__hmm_p_stress",
      "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
      "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
      "EWQ_France_zscore_60d",
      "EWQ_France_ret_20d",
      "DHR_ret_1d",
      "MRK_Merck_zscore_60d",
      "LMT_LockheedMartin_vol_20d",
      "XLB_Materials_zscore_60d",
      "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
      "vix_vs_ma20__div__spx_drawdown_252d",
      "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
      "IYM_BasicMaterials_ret_20d",
      "QQQ_vol_20d",
      "SCHW_Schwab_ret_5d",
      "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
      "EWM_Malaysia_ret_1d",
      "VIX_Price_zscore_60d__div__Russell_Price_ret_5d",
      "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d"
    ]
  },
  {
    "model_id": "egarch_v2_1j_STRESS_LogisticRegression",
    "source": "egarch_v2",
    "algo": "LogisticRegression",
    "regime": "STRESS",
    "horizon": 1,
    "F1_dir": 0.5745,
    "F1_UP_FORT": 0.1972,
    "F1_DOWN_FORT": 0.526,
    "n_features": 12,
    "train_start": "2001-02-06",
    "sampler": "BorderlineSMOTE",
    "best_params": "{}",
    "features": [
      "heston_xi__prod__hmm_p_stress",
      "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
      "HangSeng_HK_vol_20d",
      "heston_var_ev_h7",
      "NFCI_ret_1d__macross__kalman_innovation",
      "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
      "US3Y_Rate_ret_5d",
      "BA_ret_1d",
      "BLK_BlackRock_zscore_60d",
      "PG_ret_1d__div__vix_vol_of_vol_5d",
      "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d",
      "SCHW_Schwab_ret_1d__macross__XLY_Disc_ret_5d"
    ]
  },
  {
    "model_id": "egarch_v2_3j_STRESS_RandomForest",
    "source": "egarch_v2",
    "algo": "RandomForest",
    "regime": "STRESS",
    "horizon": 3,
    "F1_dir": 0.5721,
    "F1_UP_FORT": 0.3657,
    "F1_DOWN_FORT": 0.5306,
    "n_features": 15,
    "train_start": "2000-11-15",
    "sampler": "BorderlineSMOTE",
    "best_params": "{}",
    "features": [
      "vix_mean_abs_ret_5d__prod__heston_xi",
      "Core_PCE_zscore_60d",
      "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
      "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
      "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
      "US7Y_Rate_ret_20d",
      "JNJ_ret_1d",
      "EQIX_Equinix_ret_5d",
      "TM_Telephone_ret_1d",
      "CVX_ret_20d__zrel__NFCI_ret_5d",
      "heston_xi__div__hmm_p_stress",
      "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
      "COST_ret_5d__div__vix_vs_ma20",
      "hmm_p_stress",
      "PFE_ret_1d"
    ]
  },
  {
    "model_id": "egarch_v1_3j_STRESS_LogisticRegression",
    "source": "egarch_v1",
    "algo": "LogisticRegression",
    "regime": "STRESS",
    "horizon": 3,
    "F1_dir": 0.567,
    "F1_UP_FORT": 0.0211,
    "F1_DOWN_FORT": 0.4709,
    "n_features": 5,
    "train_start": "2000-11-16",
    "sampler": "BorderlineSMOTE",
    "best_params": "{}",
    "features": [
      "vix_vs_ma20__macross__vix_zscore_10d",
      "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d",
      "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d",
      "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d",
      "hmm_p_stress"
    ]
  },
  {
    "model_id": "egarch_v1_1j_GLOBAL_RandomForest",
    "source": "egarch_v1",
    "algo": "RandomForest",
    "regime": "GLOBAL",
    "horizon": 1,
    "F1_dir": 0.5649,
    "F1_UP_FORT": 0.2227,
    "F1_DOWN_FORT": 0.4468,
    "n_features": 18,
    "train_start": "2000-11-10",
    "sampler": "SMOTETomek",
    "best_params": "{}",
    "features": [
      "NFCI_ret_5d__div__IWM_SmallCap_vol_20d",
      "SPY_zscore_60d",
      "heston_xi__prod__vix_mean_abs_ret_5d",
      "SJM_JM_Smucker_ret_1d",
      "vix_momentum_3d__zrel__VZ_ret_5d",
      "SBUX_zscore_60d",
      "IWM_SmallCap_vol_20d",
      "SLB_Schlumberger_ret_5d",
      "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d",
      "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d",
      "heston_xi__minus__EBAY_eBay_ret_1d",
      "NFCI_ret_5d__div__vix_mean_abs_ret_5d",
      "EXC_Exelon_ret_1d",
      "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d",
      "NFCI_ret_5d__ret5x__vix_momentum_3d",
      "heston_xi__div__COST_vol_20d",
      "US1Y_Rate_ret_5d",
      "IWM_SmallCap_vol_20d__div__BAC_ret_1d"
    ]
  },
  {
    "model_id": "egarch_v2_5j_NORMAL_RandomForest",
    "source": "egarch_v2",
    "algo": "RandomForest",
    "regime": "NORMAL",
    "horizon": 5,
    "F1_dir": 0.5635,
    "F1_UP_FORT": 0.2835,
    "F1_DOWN_FORT": 0.4449,
    "n_features": 8,
    "train_start": "2001-02-06",
    "sampler": "BorderlineSMOTE",
    "best_params": "{}",
    "features": [
      "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
      "VIX_Price_zscore_60d__div__heston_theta",
      "VIX_Price_zscore_60d__div__hmm_p_stress",
      "EWY_Korea_ret_20d",
      "M_Macys_vol_20d",
      "Core_CPI_zscore_60d__macross__VRP",
      "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
      "NFCI_ret_5d"
    ]
  },
  {
    "model_id": "egarch_v2_5j_STRESS_LightGBM",
    "source": "egarch_v2",
    "algo": "LightGBM",
    "regime": "STRESS",
    "horizon": 5,
    "F1_dir": 0.5577,
    "F1_UP_FORT": 0.2597,
    "F1_DOWN_FORT": 0.4961,
    "n_features": 18,
    "train_start": "2000-11-15",
    "sampler": "SMOTETomek",
    "best_params": "{}",
    "features": [
      "VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d",
      "hmm_p_stress",
      "SBUX_ret_20d__minus__VIX_Price_ret_5d",
      "AAPL_ret_5d__minus__vix_vol_of_vol_10d",
      "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d",
      "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d",
      "AAPL_ret_5d__zrel__vix_vol_of_vol_10d",
      "DOW_Price_zscore_60d",
      "ORCL_zscore_60d",
      "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d",
      "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d",
      "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
      "EWM_Malaysia_zscore_60d",
      "T_ret_1d",
      "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d",
      "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d",
      "GILD_Gilead_ret_20d",
      "TGT_Target_zscore_60d"
    ]
  },
  {
    "model_id": "egarch_v1_5j_NORMAL_XGBoost",
    "source": "egarch_v1",
    "algo": "XGBoost",
    "regime": "NORMAL",
    "horizon": 5,
    "F1_dir": 0.5429,
    "F1_UP_FORT": 0.3093,
    "F1_DOWN_FORT": 0.3125,
    "n_features": 7,
    "train_start": "2001-02-07",
    "sampler": "BorderlineSMOTE",
    "best_params": "{}",
    "features": [
      "NFCI_ret_5d__minus__NFCI_vol_20d",
      "heston_var_ev_h1__prod__vix_max_abs_ret_5d",
      "BTI_BritishAmerican_ret_20d",
      "heston_var_ev_h1__minus__M_Macys_vol_20d",
      "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d",
      "EWA_Australia_ret_1d",
      "AVB_AvalonBay_zscore_60d"
    ]
  },
  {
    "model_id": "egarch_v1_3j_NORMAL_XGBoost",
    "source": "egarch_v1",
    "algo": "XGBoost",
    "regime": "NORMAL",
    "horizon": 3,
    "F1_dir": 0.5429,
    "F1_UP_FORT": 0.2948,
    "F1_DOWN_FORT": 0.2424,
    "n_features": 8,
    "train_start": "2007-01-05",
    "sampler": "SMOTE",
    "best_params": "{}",
    "features": [
      "NFCI_ret_5d__zrel__EQR_Equity_ret_1d",
      "heston_var_ev_h5__macross__SRE_Sempra_ret_5d",
      "US6M_Rate_ret_20d",
      "VVIX_ret_1d__div__vix_acceleration_1d",
      "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d",
      "NFCI_ret_20d__ret5x__VIX_Price_ret_5d",
      "heston_var_ev_h5__ret5x__HD_zscore_60d",
      "NFCI_ret_5d__div__DAX_Germany_vol_20d"
    ]
  },
  {
    "model_id": "egarch_v2_1j_NORMAL_RandomForest",
    "source": "egarch_v2",
    "algo": "RandomForest",
    "regime": "NORMAL",
    "horizon": 1,
    "F1_dir": 0.5395,
    "F1_UP_FORT": 0.3167,
    "F1_DOWN_FORT": 0.4103,
    "n_features": 5,
    "train_start": "2000-11-02",
    "sampler": "BorderlineSMOTE",
    "best_params": "{}",
    "features": [
      "heston_var_ev_h1__prod__heston_xi",
      "SBUX_vol_20d",
      "US5Y_Rate_ret_5d",
      "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
      "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d"
    ]
  },
  {
    "model_id": "egarch_v1_7j_STRESS_LogisticRegression",
    "source": "egarch_v1",
    "algo": "LogisticRegression",
    "regime": "STRESS",
    "horizon": 7,
    "F1_dir": 0.5376,
    "F1_UP_FORT": 0.219,
    "F1_DOWN_FORT": 0.37,
    "n_features": 7,
    "train_start": "2001-02-07",
    "sampler": "BorderlineSMOTE",
    "best_params": "{}",
    "features": [
      "SBUX_ret_5d__div__heston_var_ev_h3",
      "vix_momentum_2d__minus__US20Y_Rate_ret_20d",
      "DE_Deere_ret_5d__minus__hmm_p_stress",
      "US3M_Rate_vol_20d__prod__hmm_p_stress",
      "DE_Deere_ret_5d__ret5x__hmm_p_stress",
      "ROST_RossStores_ret_5d__minus__DE_Deere_ret_5d",
      "HUM_Humana_zscore_60d__div__US20Y_Rate_ret_20d"
    ]
  }
]

print(f"{len(REFERENCE_MODELS)} modèles de référence chargés")
for h in sorted(set(m['horizon'] for m in REFERENCE_MODELS)):
    n = sum(1 for m in REFERENCE_MODELS if m['horizon']==h)
    best = max((m for m in REFERENCE_MODELS if m['horizon']==h), key=lambda x: x['F1_dir'])
    print(f"  h={h}j : {n} modèles | meilleur = {best['algo']} {best['regime']} F1={best['F1_dir']}")


In [ ]:
# =============================================================================
# CHARGEMENT DES DONNÉES
# =============================================================================
def load_data(start=CONFIG['start_date']):
    t0 = time.time()
    raw = yf.download(YF_TICKERS, start=start, auto_adjust=True, progress=False)['Close']
    raw.columns = [c.replace('^','IDX_').replace('-','_') for c in raw.columns]
    coverage = raw.notna().mean()
    raw = raw.loc[:, coverage >= 0.90].ffill().dropna(how='all')
    fred_frames = []
    for name, sid in FRED_SERIES.items():
        try:
            s = web.DataReader(sid,'fred',start).squeeze(); s.name=f'FRED_{name}'
            fred_frames.append(s)
        except Exception as e: print(f"  [WARN] FRED {sid}: {e}")
    if fred_frames:
        raw = pd.concat([raw, pd.concat(fred_frames,axis=1).reindex(raw.index,method='ffill')], axis=1)
    print(f"  Dataset: {raw.shape[0]}j × {raw.shape[1]} séries ({time.time()-t0:.1f}s)")
    return raw

df_raw = load_data()

# Split 80/20
all_dates  = df_raw.dropna(how='all').index.sort_values()
split_idx  = int(len(all_dates)*0.80)
TEST_DATE  = all_dates[split_idx].strftime('%Y-%m-%d')
CONFIG['test_date'] = TEST_DATE
print(f"  Split 80/20 → Train: {all_dates[split_idx-1].date()} | Test: {all_dates[split_idx].date()}")


In [ ]:
# =============================================================================
# FEATURE ENGINEERING — Features disponibles pour tous les modèles
# Inclut toutes les features référencées dans REFERENCE_MODELS
# =============================================================================
def build_all_features(df_raw, train_end_idx):
    t0 = time.time()
    feats = pd.DataFrame(index=df_raw.index)

    # Colonnes VIX et SPX
    vix_col = [c for c in df_raw.columns if ('IDX_VIX' in c or c.endswith('_VIX'))
                and 'VXN' not in c and 'VVIX' not in c][0]
    spx_cols = [c for c in df_raw.columns if 'GSPC' in c or c=='SPY']
    spx_col  = spx_cols[0] if spx_cols else None
    vix = df_raw[vix_col].replace([np.inf,-np.inf],np.nan).ffill().bfill()
    vix_ret = np.log(vix/vix.shift(1)).replace([np.inf,-np.inf],np.nan).fillna(0)

    # ── Rendements et vol pour chaque ticker ─────────────────────────────────
    for col in df_raw.columns:
        s = df_raw[col].ffill().bfill()
        for w in [1,5,20]:
            ret = np.log(s/s.shift(w)).replace([np.inf,-np.inf],np.nan)
            feats[f'{col}_ret_{w}d']  = ret
        feats[f'{col}_vol_20d']      = np.log(s/s.shift(1)).rolling(20,min_periods=10).std()
        feats[f'{col}_zscore_60d']   = (s - s.rolling(60,min_periods=30).mean()) /                                         s.rolling(60,min_periods=30).std().replace(0,np.nan)

    # ── VIX features spécifiques ──────────────────────────────────────────────
    feats['vix_level']          = vix
    feats['vix_ma_20']          = vix.rolling(20,min_periods=10).mean()
    feats['vix_vs_ma5']         = (vix - vix.rolling(5,min_periods=3).mean()) /                                    vix.rolling(5,min_periods=3).mean().replace(0,np.nan)
    feats['vix_vs_ma10']        = (vix - vix.rolling(10,min_periods=5).mean()) /                                    vix.rolling(10,min_periods=5).mean().replace(0,np.nan)
    feats['vix_zscore_10d']     = (vix - vix.rolling(10,min_periods=5).mean()) /                                    vix.rolling(10,min_periods=5).std().replace(0,np.nan)
    feats['vix_vol_of_vol_5d']  = vix_ret.rolling(5,min_periods=3).std()
    feats['vix_vol_of_vol_10d'] = vix_ret.rolling(10,min_periods=5).std()
    feats['vix_momentum_3d']    = vix.pct_change(3)
    feats['vix_acceleration_1d']= vix_ret - vix_ret.shift(1)
    feats['vix_acceleration_3d']= vix_ret - vix_ret.shift(3)
    feats['vix_erratic_ratio']  = vix_ret.abs().rolling(5,min_periods=3).max() /                                    vix_ret.abs().rolling(5,min_periods=3).mean().replace(0,np.nan)
    feats['vix_vol_ratio_5_60'] = vix_ret.rolling(5,min_periods=3).std() /                                    vix_ret.rolling(60,min_periods=30).std().replace(0,np.nan)
    feats['vix_max_abs_ret_5d'] = vix_ret.abs().rolling(5,min_periods=3).max()
    feats['vix_mean_abs_ret_5d']= vix_ret.abs().rolling(5,min_periods=3).mean()

    # ── SPX drawdown ──────────────────────────────────────────────────────────
    if spx_col:
        spx = df_raw[spx_col].ffill().bfill()
        roll_high = spx.rolling(252,min_periods=126).max()
        feats['spx_drawdown_252d'] = (spx-roll_high)/roll_high.replace(0,np.nan)
        spx_ret = np.log(spx/spx.shift(1)).fillna(0)
        feats['spx_vol_5d'] = spx_ret.rolling(5,min_periods=3).std()
        feats['spx_abs_ret_max_5d'] = spx_ret.abs().rolling(5,min_periods=3).max()
        feats['spx_momentum_3d'] = spx.pct_change(3)

    # ── EGARCH SPX ────────────────────────────────────────────────────────────
    if spx_col:
        try:
            spx_ret_tr = spx_ret.iloc[:train_end_idx]*100
            am  = arch_model(spx_ret_tr, vol='EGARCH', p=1, q=1, dist='skewt', rescale=False)
            res = am.fit(disp='off', show_warning=False)
            fc  = res.forecast(start=0, reindex=True)
            cv  = (fc.variance.iloc[:,0]/10000).reindex(df_raw.index, method='ffill')
            cv  = cv.replace([np.inf,-np.inf],np.nan).ffill().bfill()
            feats['egarch_condvar'] = cv
            feats['egarch_delta']   = cv.diff()
            for h in [1,3,5]:
                feats[f'EGARCH_SPX_condvar_h{h}'] = cv
                feats[f'EGARCH_SPX_delta_h{h}']   = cv.diff()
            print(f"  EGARCH OK ({time.time()-t0:.1f}s)")
        except Exception as e: print(f"  [WARN] EGARCH: {e}")

    # ── Kalman (avec shift anti-leakage) ─────────────────────────────────────
    try:
        vc  = vix.interpolate('linear').ffill().bfill().astype(float)
        kf  = KalmanFilter(transition_matrices=[[1.]], observation_matrices=[[1.]],
                           initial_state_mean=[float(vc.iloc[0])],
                           initial_state_covariance=[[1.]],
                           transition_covariance=[[1e-3]], observation_covariance=[[1e-2]],
                           em_vars=['transition_covariance','observation_covariance'])
        kf  = kf.em(vc.iloc[:train_end_idx].values.reshape(-1,1), n_iter=20)
        sm, _ = kf.filter(vc.values.reshape(-1,1))
        ss, _ = kf.smooth(vc.values.reshape(-1,1))
        kf_f  = pd.Series(sm[:,0], index=df_raw.index)
        kf_s  = pd.Series(ss[:,0], index=df_raw.index)
        feats['VIX_Residual']   = (vc-kf_f).shift(1).replace([np.inf,-np.inf],np.nan)
        feats['VIX_Innovation'] = (vc-kf_s.shift(1)).shift(1).replace([np.inf,-np.inf],np.nan)
        feats['kalman_residual']   = feats['VIX_Residual']
        feats['kalman_innovation'] = feats['VIX_Innovation']
        print(f"  Kalman OK ({time.time()-t0:.1f}s)")
    except Exception as e: print(f"  [WARN] Kalman: {e}")

    # ── HMM ───────────────────────────────────────────────────────────────────
    try:
        rv5  = vix_ret.pow(2).rolling(5,min_periods=3).mean()
        mu_t = vix.iloc[:train_end_idx].mean(); sd_t = vix.iloc[:train_end_idx].std()
        vix_n = (vix - mu_t) / (sd_t if sd_t>1e-8 else 1)
        X_hmm = pd.DataFrame({'r':vix_ret,'v':np.sqrt(rv5),'l':vix_n}).dropna()
        X_tr  = X_hmm.iloc[:train_end_idx].values
        mh    = hmmlib.GaussianHMM(n_components=2, covariance_type='full',
                                    n_iter=200, random_state=SEED)
        mh.fit(X_tr)
        states = mh.predict(X_tr)
        sv     = [rv5.reindex(X_hmm.index[:train_end_idx]).values[states==s].mean() for s in range(2)]
        ss_idx = int(np.argmax(sv))
        proba  = mh.predict_proba(X_hmm.values)
        feats['P_stress_HMM'] = pd.Series(proba[:,ss_idx], index=X_hmm.index).reindex(df_raw.index)
        feats['hmm_p_stress'] = feats['P_stress_HMM']
        print(f"  HMM OK ({time.time()-t0:.1f}s)")
    except Exception as e: print(f"  [WARN] HMM: {e}")

    # ── Heston proxies ────────────────────────────────────────────────────────
    try:
        v0 = (vix/100).pow(2)
        theta = vix_ret.pow(2).rolling(60,min_periods=30).mean()
        vvix_c = [c for c in df_raw.columns if 'VVIX' in c]
        xi  = df_raw[vvix_c[0]].ffill().bfill()/100 if vvix_c else vix_ret.rolling(20).std()
        xi  = xi.reindex(df_raw.index, method='ffill')
        if spx_col:
            rho = vix_ret.rolling(30,min_periods=15).corr(spx_ret)
        else:
            rho = pd.Series(-0.7, index=df_raw.index)

        def rolling_kappa(s, w=252):
            k = pd.Series(np.nan, index=s.index)
            sf = s.ffill().bfill()
            for i in range(w, len(sf)):
                y_ = sf.iloc[i-w+1:i+1].values; x_ = sf.iloc[i-w:i].values
                try:
                    b = np.corrcoef(x_,y_)[0,1]
                    if np.isfinite(b) and 0<abs(b)<0.9999:
                        hl = -np.log(2)/np.log(abs(b))
                        if np.isfinite(hl) and hl>0: k.iloc[i] = np.log(2)/hl
                except: pass
            return k.replace([np.inf,-np.inf],np.nan)

        kappa = rolling_kappa(vix)
        feats['heston_v0']    = v0
        feats['heston_theta'] = theta
        feats['heston_xi']    = xi
        feats['heston_rho']   = rho
        feats['heston_kappa'] = kappa
        feats['heston_feller']= (2*kappa*theta)/xi.pow(2).replace(0,np.nan)
        feats['heston_v0_minus_theta'] = v0-theta
        for h in [1,3,5,7]:
            ev = theta + (v0-theta)*np.exp(-kappa*h)
            feats[f'heston_ev_h{h}']     = ev.replace([np.inf,-np.inf],np.nan)
            feats[f'heston_spread_h{h}'] = v0-ev
            feats[f'heston_vol_h{h}']    = np.sqrt(ev.clip(lower=0))*100
        print(f"  Heston OK ({time.time()-t0:.1f}s)")
    except Exception as e: print(f"  [WARN] Heston: {e}")

    # ── VRP ───────────────────────────────────────────────────────────────────
    try:
        import statsmodels.api as sm
        rv1d = vix_ret.pow(2); rv5d = rv1d.rolling(5,min_periods=3).mean()
        rv22d = rv1d.rolling(22,min_periods=10).mean()
        rv_tgt = rv1d.shift(-22).rolling(22,min_periods=11).mean()
        hdf = pd.DataFrame({'rv1':rv1d,'rv5':rv5d,'rv22':rv22d,'y':rv_tgt}).dropna()
        hdf = hdf.replace([np.inf,-np.inf],np.nan).dropna()
        htr = hdf.iloc[:train_end_idx]
        Xh  = sm.add_constant(htr[['rv1','rv5','rv22']], has_constant='add')
        hm  = sm.OLS(htr['y'], Xh).fit()
        Xf  = sm.add_constant(hdf[['rv1','rv5','rv22']], has_constant='add').fillna(0)
        rv_pred = hm.predict(Xf).reindex(df_raw.index).fillna(0)
        feats['VRP'] = (vix/100).pow(2) - rv_pred
        mu_v = feats['VRP'].iloc[:train_end_idx].mean()
        sd_v = feats['VRP'].iloc[:train_end_idx].std()
        feats['VRP_zscore'] = (feats['VRP']-mu_v)/(sd_v if sd_v>1e-8 else 1)
        feats['VRP_ma5'] = feats['VRP'].rolling(5,min_periods=3).mean()
    except Exception as e: print(f"  [WARN] VRP: {e}")

    # ── Jump Intensity + Hawkes ───────────────────────────────────────────────
    sig60 = vix_ret.rolling(60,min_periods=30).std()
    is_j  = (vix_ret.abs()>3*sig60).astype(float)
    feats['jump_intensity_20d'] = is_j.rolling(20,min_periods=10).mean()
    feats['jump_intensity_60d'] = is_j.rolling(60,min_periods=30).mean()
    try:
        sig_hw = vix_ret.rolling(30,min_periods=15).std()
        jt     = vix_ret.index[vix_ret.abs()>2*sig_hw]
        hw     = pd.Series(0., index=vix_ret.index)
        for i,t in enumerate(vix_ret.index):
            past = jt[jt<t]
            hw.iloc[i] = 0.3 + 0.3*np.sum(np.exp(-0.1*np.array([(t-tj).days for tj in past]))) if len(past) else 0.3
        mu_hw = hw.iloc[:train_end_idx].mean(); sd_hw = hw.iloc[:train_end_idx].std()
        feats['hawkes_intensity'] = hw
        feats['hawkes_zscore']    = (hw-mu_hw)/(sd_hw if sd_hw>1e-8 else 1)
        print(f"  VRP+Hawkes OK ({time.time()-t0:.1f}s)")
    except Exception as e: print(f"  [WARN] Hawkes: {e}")

    feats = feats.replace([np.inf,-np.inf], np.nan)
    print(f"  Total features engineered: {feats.shape[1]} ({time.time()-t0:.1f}s)")
    return feats

print("[Features] Calcul en cours...")
ts_feats = build_all_features(df_raw, split_idx)
df_features = pd.concat([df_raw, ts_feats], axis=1).replace([np.inf,-np.inf], np.nan)
print(f"Dataset complet: {df_features.shape}")


In [ ]:
# =============================================================================
# CIBLE AMPLITUDE — quantiles conditionnels par régime, calculés sur le train
# =============================================================================
def build_target(vix_series, horizon, train_end_idx, df_all):
    vix    = vix_series.ffill().bfill()
    vix_tr = vix.iloc[:train_end_idx]
    calm_thr   = vix_tr.quantile(0.33)
    stress_thr = vix_tr.quantile(0.67)

    regime = pd.Series('NORMAL', index=vix.index)
    regime[vix < calm_thr]    = 'CALM'
    regime[vix >= stress_thr] = 'STRESS'

    ret  = (vix.shift(-horizon)/vix) - 1
    flat = ret.abs() < CONFIG['flat_thr']
    ret  = ret.loc[~flat].dropna()
    regime_ret = regime.reindex(ret.index)

    ret_tr = ret.iloc[:train_end_idx]
    thresholds = {}
    for reg in ['CALM','NORMAL','STRESS']:
        sub = ret_tr[regime_ret.iloc[:train_end_idx]==reg]
        thresholds[reg] = (sub.quantile(0.25) if len(sub)>=20 else ret_tr.quantile(0.25),
                           sub.quantile(0.75) if len(sub)>=20 else ret_tr.quantile(0.75))
    thresholds['GLOBAL'] = (ret_tr.quantile(0.25), ret_tr.quantile(0.75))

    def classify(r, reg):
        q25,q75 = thresholds.get(reg,(0,0))
        if r<q25: return 0
        if r<0:   return 1
        if r<q75: return 2
        return 3

    target = pd.Series([classify(r,regime_ret[i]) for i,r in ret.items()],
                        index=ret.index, name=TARGET_COL)
    return target, regime_ret, thresholds

# Identifier colonnes VIX et split
vix_col  = [c for c in df_raw.columns if ('IDX_VIX' in c or c.endswith('_VIX'))
             and 'VXN' not in c and 'VVIX' not in c][0]
all_dates = df_raw.dropna(how='all').index.sort_values()
split_idx = int(len(all_dates)*0.80)

print(f"VIX col: {vix_col}")
print(f"Split: train → {all_dates[split_idx-1].date()} | test → {all_dates[split_idx].date()}")


In [ ]:
# =============================================================================
# RECONSTRUCTION DES INTERACTIONS
# Les noms de features de la forme "A__op__B" sont reconstruits ici
# =============================================================================
SEPS = {'__div__':'div','__minus__':'minus','__prod__':'prod',
        '__zrel__':'zrel','__macross__':'macross','__ret5x__':'ret5x',
        # anciens formats
        '__div__':'div','__minus__':'minus'}

def reconstruct_feature(fname, df, rolling_w=20, eps=1e-8):
    """Reconstruit une feature d'interaction depuis son nom."""
    if fname in df.columns:
        return df[fname]
    for sep, op in [('__div__','div'),('__minus__','minus'),('__prod__','prod'),
                    ('__zrel__','zrel'),('__macross__','macross'),('__ret5x__','ret5x')]:
        if sep in fname:
            a,b = fname.split(sep,1)
            if a not in df.columns or b not in df.columns:
                return None
            si,sj = df[a], df[b]
            if op=='div':     return si/sj.where(sj.abs()>=eps, np.nan)
            if op=='minus':   return si-sj
            if op=='prod':    return si*sj
            if op=='zrel':
                d = si-sj; rs = d.rolling(rolling_w,min_periods=rolling_w//2).std()
                return d/rs.replace(0,np.nan)
            if op=='macross':
                mai = si.rolling(rolling_w,min_periods=rolling_w//2).mean()
                maj = sj.rolling(rolling_w,min_periods=rolling_w//2).mean()
                return mai/maj.where(maj.abs()>=eps,np.nan)
            if op=='ret5x':   return si.pct_change(5)*sj
    return None


def build_feature_matrix(df, feature_names, rolling_w=20):
    """Construit la matrice de features (base + interactions) pour un modèle donné."""
    cols = {}
    missing = []
    for fname in feature_names:
        s = reconstruct_feature(fname, df, rolling_w=rolling_w)
        if s is not None:
            cols[fname] = s
        else:
            missing.append(fname)
    if missing:
        print(f"  [WARN] {len(missing)} features introuvables: {missing[:3]}...")
    if not cols:
        return pd.DataFrame(index=df.index)
    result = pd.DataFrame(cols, index=df.index).replace([np.inf,-np.inf], np.nan)
    return result


In [ ]:
# =============================================================================
# MAPPING ALGORITHMES + SAMPLER ADAPTATIF
# =============================================================================
def get_algo(algo_name, n_features, best_params_str='{}'):
    """Instancie l'algorithme avec les bons hyperparamètres."""
    try:
        params = json.loads(best_params_str) if best_params_str not in ('{}','nan','None') else {}
    except: params = {}

    if 'XGBoost' in algo_name:
        defaults = dict(n_estimators=200, max_depth=4, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                        eval_metric='mlogloss', objective='multi:softprob',
                        random_state=SEED, n_jobs=-1)
        defaults.update({k:v for k,v in params.items()
                          if k in ['n_estimators','max_depth','learning_rate',
                                   'subsample','colsample_bytree','min_child_weight']})
        return XGBClassifier(**defaults)

    if 'LightGBM' in algo_name:
        defaults = dict(n_estimators=200, max_depth=5, learning_rate=0.05,
                        num_leaves=31, subsample=0.8, min_child_samples=10,
                        class_weight='balanced', random_state=SEED, verbose=-1, n_jobs=-1)
        defaults.update({k:v for k,v in params.items()
                          if k in ['n_estimators','max_depth','learning_rate',
                                   'num_leaves','min_child_samples']})
        return LGBMClassifier(**defaults)

    if 'GradientBoosting' in algo_name:
        defaults = dict(n_estimators=200, learning_rate=0.05, max_depth=4,
                        min_samples_leaf=10, subsample=0.8, random_state=SEED)
        defaults.update({k:v for k,v in params.items()
                          if k in ['n_estimators','learning_rate','max_depth',
                                   'min_samples_leaf','subsample']})
        return GradientBoostingClassifier(**defaults)

    if 'RandomForest' in algo_name:
        defaults = dict(n_estimators=200, max_depth=6, min_samples_leaf=5,
                        class_weight='balanced', random_state=SEED, n_jobs=-1)
        defaults.update({k:v for k,v in params.items()
                          if k in ['n_estimators','max_depth','min_samples_leaf']})
        return RandomForestClassifier(**defaults)

    if 'LogisticRegression' in algo_name:
        defaults = dict(C=1.0, max_iter=2000, class_weight='balanced',
                        multi_class='multinomial', solver='lbfgs', random_state=SEED)
        defaults.update({k:v for k,v in params.items() if k in ['C']})
        return LogisticRegression(**defaults)

    raise ValueError(f"Algo inconnu: {algo_name}")


def get_sampler(sampler_name):
    mapping = {
        'SMOTE':           SMOTE(random_state=SEED),
        'BorderlineSMOTE': BorderlineSMOTE(random_state=SEED, kind='borderline-1'),
        'SMOTETomek':      SMOTETomek(random_state=SEED),
        'ADASYN':          SMOTE(random_state=SEED),  # fallback ADASYN → SMOTE
    }
    return mapping.get(sampler_name, SMOTE(random_state=SEED))


def compute_metrics(y_true, y_pred, y_prob=None):
    """Métriques hiérarchiques L1/L2/L3."""
    dm = {0:'DOWN',1:'DOWN',2:'UP',3:'UP'}
    yd_t = [dm[y] for y in y_true]; yd_p = [dm[y] for y in y_pred]
    m = {
        'F1_4cls':  f1_score(y_true,y_pred,average='macro',zero_division=0),
        'Acc_4cls': accuracy_score(y_true,y_pred),
        'Acc_dir':  accuracy_score(yd_t,yd_p),
        'F1_dir':   f1_score(yd_t,yd_p,average='macro',zero_division=0),
        'F1_UP':    f1_score(yd_t,yd_p,pos_label='UP',  average='binary',zero_division=0),
        'F1_DOWN':  f1_score(yd_t,yd_p,pos_label='DOWN',average='binary',zero_division=0),
    }
    ui = [i for i,y in enumerate(y_true) if dm[y]=='UP']
    di = [i for i,y in enumerate(y_true) if dm[y]=='DOWN']
    if ui:
        yt=['FORT' if y_true[i]==3 else 'FAIBLE' for i in ui]
        yp=['FORT' if y_pred[i]==3 else 'FAIBLE' for i in ui]
        m['F1_UP_FORT']   = f1_score(yt,yp,pos_label='FORT',average='binary',zero_division=0)
    if di:
        yt=['FORT' if y_true[i]==0 else 'FAIBLE' for i in di]
        yp=['FORT' if y_pred[i]==0 else 'FAIBLE' for i in di]
        m['F1_DOWN_FORT'] = f1_score(yt,yp,pos_label='FORT',average='binary',zero_division=0)
    return m


In [ ]:
# =============================================================================
# ENTRAÎNEMENT DES 28 MODÈLES DE RÉFÉRENCE
# + HORIZONS MANQUANTS 2j et 10j (features héritées du SHAP Phase1 le plus proche)
# =============================================================================
trained_models   = {}   # {model_id: fitted_clf}
model_scalers    = {}   # {model_id: RobustScaler}
model_features   = {}   # {model_id: liste de features}
model_targets    = {}   # {horizon: (y_train, y_test, idx_train, idx_test)}
model_metrics    = {}   # {model_id: dict métriques}
model_proba_test = {}   # {model_id: array (N_test, 4)}
model_meta       = {}   # {model_id: dict config}

t_total = time.time()

# Horizons manquants : 2j et 10j n'ont pas de modèles dans les runs précédents
# → on réutilise les features des horizons les plus proches (1j et 7j)
HORIZON_FEATURE_MAP = {2: 1, 10: 7}  # h manquant → h source pour les features

# Construire les targets pour tous les horizons
targets_by_h = {}
for h in CONFIG['horizons']:
    tgt, reg_s, thr = build_target(df_features[vix_col], h, split_idx, df_features)
    targets_by_h[h] = {'target': tgt, 'regime': reg_s, 'thresholds': thr}
    print(f"  h={h}j: {len(tgt)} obs | {tgt.value_counts().to_dict()}")

# Entraîner les modèles de référence (h=1,3,5,7)
for cfg in REFERENCE_MODELS:
    model_id = cfg['model_id']
    h        = cfg['horizon']
    regime   = cfg['regime']
    algo     = cfg['algo']
    features = cfg['features']
    ts       = cfg['train_start']
    sampler  = cfg['sampler']

    t0 = time.time()
    print(f"  Training {model_id}...", end=' ')

    # Target
    tgt_info = targets_by_h[h]
    target   = tgt_info['target']
    reg_s    = tgt_info['regime']

    # Construire la matrice de features
    df_aligned = df_features.reindex(target.index)
    X_mat = build_feature_matrix(df_aligned, features)
    if X_mat.empty or X_mat.shape[1] == 0:
        print(f"[SKIP] pas de features")
        continue

    # Ajouter la target
    df_aligned = df_aligned.copy(); df_aligned[TARGET_COL] = target
    X_mat[TARGET_COL] = target

    # Filtrer par régime et split train/test
    tr_mask = X_mat.index < pd.Timestamp(TEST_DATE)
    te_mask = X_mat.index >= pd.Timestamp(TEST_DATE)
    ts_mask = X_mat.index >= pd.Timestamp(ts) if ts else slice(None)

    df_tr = X_mat.loc[tr_mask & (X_mat.index >= pd.Timestamp(ts))].dropna(subset=[TARGET_COL])
    df_te = X_mat.loc[te_mask].dropna(subset=[TARGET_COL])
    if regime != 'GLOBAL':
        df_tr = df_tr.loc[reg_s.reindex(df_tr.index)==regime]
        df_te = df_te.loc[reg_s.reindex(df_te.index)==regime]

    feat_cols = [c for c in X_mat.columns if c != TARGET_COL]
    if len(df_tr) < 30 or len(df_te) < 5:
        print(f"[SKIP] train={len(df_tr)} test={len(df_te)}")
        continue

    y_tr = df_tr[TARGET_COL].values.astype(int)
    y_te = df_te[TARGET_COL].values.astype(int)
    X_tr = df_tr[feat_cols].fillna(0).values
    X_te = df_te[feat_cols].fillna(0).values

    # Scale
    sc = RobustScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc = sc.transform(X_te)

    # SMOTE
    try:
        samp = get_sampler(sampler)
        X_res, y_res = samp.fit_resample(X_tr_sc, y_tr)
    except:
        X_res, y_res = X_tr_sc, y_tr

    # Fit
    try:
        clf = get_algo(algo, len(feat_cols), cfg.get('best_params','{}'))
        clf.fit(X_res, y_res)
    except Exception as e:
        print(f"[ERR] {e}"); continue

    # Évaluation
    y_pred = clf.predict(X_te_sc)
    y_prob = clf.predict_proba(X_te_sc) if hasattr(clf,'predict_proba') else None

    # Aligner les proba sur 4 classes
    if y_prob is not None:
        if y_prob.shape[1] < 4:
            full_prob = np.zeros((len(y_prob),4))
            for ci, c in enumerate(clf.classes_):
                full_prob[:,int(c)] = y_prob[:,ci]
            y_prob = full_prob

    met = compute_metrics(y_te, y_pred, y_prob)
    print(f"F1_dir={met['F1_dir']:.4f} ({time.time()-t0:.1f}s)")

    trained_models[model_id]   = clf
    model_scalers[model_id]    = sc
    model_features[model_id]   = feat_cols
    model_metrics[model_id]    = {**met, **{k:cfg[k] for k in ['horizon','regime','algo','F1_dir','train_start']}}
    model_proba_test[model_id] = y_prob
    model_meta[model_id]       = cfg

print(f"\n[DONE] {len(trained_models)} modèles entraînés en {time.time()-t_total:.1f}s")


In [ ]:
# =============================================================================
# HORIZONS MANQUANTS : 2j et 10j
# Features héritées des horizons les plus proches (1j → 2j, 7j → 10j)
# Algorithmes : les meilleurs de chaque régime sur les horizons voisins
# =============================================================================
NEW_HORIZONS = {
    2:  {'source_h': 1,  'regimes': ['CALM','NORMAL','STRESS','GLOBAL']},
    10: {'source_h': 7,  'regimes': ['CALM','NORMAL','STRESS','GLOBAL']},
}

for h_new, cfg_new in NEW_HORIZONS.items():
    h_src = cfg_new['source_h']
    t0 = time.time()
    print(f"\n=== Horizons {h_new}j (features héritées de {h_src}j) ===")

    tgt_info = targets_by_h[h_new]
    target   = tgt_info['target']
    reg_s    = tgt_info['regime']

    # Prendre les features des meilleurs modèles du h source pour chaque régime
    src_models = [m for m in REFERENCE_MODELS
                  if m['horizon']==h_src and m['F1_dir']>=0.55]
    if not src_models:
        src_models = [m for m in REFERENCE_MODELS if m['horizon']==h_src]

    for regime in cfg_new['regimes']:
        # Choisir le meilleur modèle source pour ce régime
        regime_models = [m for m in src_models if m['regime']==regime]
        if not regime_models:
            regime_models = [m for m in src_models if m['regime']=='GLOBAL']
        if not regime_models: continue
        best_src = max(regime_models, key=lambda x: x['F1_dir'])
        features = best_src['features']
        algo     = best_src['algo']

        model_id = f"h{h_new}j_{regime}_{algo}"
        print(f"  {model_id} (features from h={h_src}j {best_src['regime']})", end=' ')

        X_mat = build_feature_matrix(df_features.reindex(target.index), features)
        if X_mat.empty: print("[SKIP]"); continue
        X_mat[TARGET_COL] = target

        tr_mask = X_mat.index <  pd.Timestamp(TEST_DATE)
        te_mask = X_mat.index >= pd.Timestamp(TEST_DATE)
        df_tr   = X_mat.loc[tr_mask].dropna(subset=[TARGET_COL])
        df_te   = X_mat.loc[te_mask].dropna(subset=[TARGET_COL])
        if regime != 'GLOBAL':
            df_tr = df_tr.loc[reg_s.reindex(df_tr.index)==regime]
            df_te = df_te.loc[reg_s.reindex(df_te.index)==regime]

        feat_cols = [c for c in X_mat.columns if c!=TARGET_COL]
        if len(df_tr)<30 or len(df_te)<5: print("[SKIP small]"); continue

        y_tr = df_tr[TARGET_COL].values.astype(int)
        y_te = df_te[TARGET_COL].values.astype(int)
        X_tr = df_tr[feat_cols].fillna(0).values
        X_te = df_te[feat_cols].fillna(0).values

        sc = RobustScaler()
        X_tr_sc = sc.fit_transform(X_tr); X_te_sc = sc.transform(X_te)
        try:
            samp = get_sampler(best_src['sampler'])
            X_res, y_res = samp.fit_resample(X_tr_sc, y_tr)
        except: X_res, y_res = X_tr_sc, y_tr

        try:
            clf = get_algo(algo, len(feat_cols))
            clf.fit(X_res, y_res)
        except Exception as e: print(f"[ERR] {e}"); continue

        y_pred = clf.predict(X_te_sc)
        y_prob = clf.predict_proba(X_te_sc) if hasattr(clf,'predict_proba') else None
        if y_prob is not None and y_prob.shape[1]<4:
            fp = np.zeros((len(y_prob),4))
            for ci,c in enumerate(clf.classes_): fp[:,int(c)] = y_prob[:,ci]
            y_prob = fp

        met = compute_metrics(y_te, y_pred, y_prob)
        print(f"F1_dir={met['F1_dir']:.4f} ({time.time()-t0:.1f}s)")

        trained_models[model_id]   = clf
        model_scalers[model_id]    = sc
        model_features[model_id]   = feat_cols
        model_metrics[model_id]    = {**met,'horizon':h_new,'regime':regime,'algo':algo,
                                       'F1_dir':met['F1_dir'],'train_start':'2000-01-01'}
        model_proba_test[model_id] = y_prob
        model_meta[model_id]       = {**best_src,'horizon':h_new,'model_id':model_id}

print(f"\n[DONE] Total modèles (incl. 2j+10j): {len(trained_models)}")


In [ ]:
# =============================================================================
# STACKING MULTI-HORIZON — Méta-modèle XGBoost
#
# Principe :
# 1. Pour chaque date t du train, générer les probabilités OOF de chaque modèle
#    (le modèle prédit sur les folds qu'il n'a pas vus → zéro leakage)
# 2. Concatener : (N_train, n_modèles × 4 classes) comme features du méta-modèle
# 3. Cible du méta-modèle : direction binaire UP/DOWN à h=5j (horizon cible principal)
#    ou amplitude 4 classes si suffisamment de données
#
# =============================================================================
META_TARGET_HORIZON = 5   # horizon de la cible du méta-modèle

print(f"=== STACKING — {len(trained_models)} modèles × 4 classes ===")
print(f"Cible méta-modèle : h={META_TARGET_HORIZON}j")

tgt_meta_info = targets_by_h[META_TARGET_HORIZON]
target_meta   = tgt_meta_info['target']

# Aligner tous les modèles sur le même index (dates communes test)
# Pour le stacking OOF, on utilise le train de chaque modèle
# → on prend l'intersection des dates disponibles

def generate_oof_probas(model_id, df_features, target, n_folds=5):
    """
    Génère les probabilités OOF pour un modèle donné.
    Out-of-Fold : le modèle est ré-entraîné sur K-1 folds et prédit sur le Kème.
    Garantit que les features du méta-modèle ne sont pas fuitées.
    """
    cfg  = model_meta[model_id]
    h    = cfg['horizon']
    reg  = cfg['regime']
    algo = cfg['algo']
    feats = model_features[model_id]

    tgt_h = targets_by_h[h]['target']
    reg_s = targets_by_h[h]['regime']

    X_mat = build_feature_matrix(df_features.reindex(tgt_h.index), feats)
    if X_mat.empty: return None, None
    X_mat[TARGET_COL] = tgt_h

    # Train uniquement
    df_tr = X_mat.loc[X_mat.index < pd.Timestamp(TEST_DATE)].dropna(subset=[TARGET_COL])
    if reg != 'GLOBAL':
        df_tr = df_tr.loc[reg_s.reindex(df_tr.index)==reg]
    if len(df_tr) < 50: return None, None

    feat_cols = [c for c in X_mat.columns if c!=TARGET_COL]
    X_tr = df_tr[feat_cols].fillna(0).values
    y_tr = df_tr[TARGET_COL].values.astype(int)

    sc = model_scalers[model_id]
    X_tr_sc = sc.transform(X_tr)

    # OOF par TimeSeriesSplit
    oof_probs = np.full((len(X_tr_sc), 4), np.nan)
    tscv = TimeSeriesSplit(n_splits=n_folds)

    for tr_idx, val_idx in tscv.split(X_tr_sc):
        if len(val_idx) < 5: continue
        try:
            samp = get_sampler(cfg.get('sampler','SMOTE'))
            X_res_f, y_res_f = samp.fit_resample(X_tr_sc[tr_idx], y_tr[tr_idx])
        except:
            X_res_f, y_res_f = X_tr_sc[tr_idx], y_tr[tr_idx]

        try:
            clf_f = get_algo(algo, len(feat_cols))
            clf_f.fit(X_res_f, y_res_f)
            probs = clf_f.predict_proba(X_tr_sc[val_idx])
            if probs.shape[1] < 4:
                fp = np.zeros((len(probs),4))
                for ci,c in enumerate(clf_f.classes_): fp[:,int(c)] = probs[:,ci]
                probs = fp
            oof_probs[val_idx] = probs
        except: pass

    return oof_probs, df_tr.index


# Générer les OOF pour tous les modèles
print("\nGénération des probabilités OOF...")
t0 = time.time()

# Index commun du train (meta target h=5j)
tr_idx_meta = target_meta.index[target_meta.index < pd.Timestamp(TEST_DATE)]
te_idx_meta = target_meta.index[target_meta.index >= pd.Timestamp(TEST_DATE)]

meta_X_train_list = []
meta_X_test_list  = []
meta_model_ids    = []

for model_id in list(trained_models.keys()):
    print(f"  OOF {model_id}...", end=' ')
    oof_probs, tr_dates = generate_oof_probas(model_id, df_features, target_meta)
    if oof_probs is None:
        print("[SKIP]"); continue

    # Aligner sur l'index commun train meta
    oof_series = pd.DataFrame(oof_probs, index=tr_dates,
                               columns=[f'{model_id}_c{k}' for k in range(4)])
    oof_aligned = oof_series.reindex(tr_idx_meta).fillna(0.25)

    # Probabilités sur le test
    test_probs = model_proba_test.get(model_id)
    if test_probs is None:
        print("[SKIP test]"); continue

    # Aligner les proba test sur l'index test meta
    cfg   = model_meta[model_id]
    h     = cfg['horizon']; reg = cfg['regime']
    tgt_h = targets_by_h[h]['target']
    reg_s = targets_by_h[h]['regime']
    te_dates = tgt_h.index[tgt_h.index >= pd.Timestamp(TEST_DATE)]
    if reg != 'GLOBAL': te_dates = te_dates[reg_s.reindex(te_dates)==reg]
    te_dates_common = te_dates[:len(test_probs)]
    test_df = pd.DataFrame(test_probs[:len(te_dates_common)], index=te_dates_common,
                            columns=[f'{model_id}_c{k}' for k in range(4)])
    test_aligned = test_df.reindex(te_idx_meta).fillna(0.25)

    meta_X_train_list.append(oof_aligned)
    meta_X_test_list.append(test_aligned)
    meta_model_ids.append(model_id)
    print(f"OK")

# Concatener
meta_X_train = pd.concat(meta_X_train_list, axis=1).fillna(0.25)
meta_X_test  = pd.concat(meta_X_test_list,  axis=1).fillna(0.25)

# Cible : direction binaire UP/DOWN (plus stable que 4 classes pour le méta-modèle)
y_meta_dir_train = (target_meta.reindex(tr_idx_meta).fillna(0).values >= 2).astype(int)
y_meta_dir_test  = (target_meta.reindex(te_idx_meta).fillna(0).values >= 2).astype(int)

# Cible 4 classes aussi (pour comparaison)
y_meta_4cls_train = target_meta.reindex(tr_idx_meta).fillna(0).values.astype(int)
y_meta_4cls_test  = target_meta.reindex(te_idx_meta).fillna(0).values.astype(int)

print(f"\n[OOF DONE] {meta_X_train.shape[1]} features | Train={len(meta_X_train)} | Test={len(meta_X_test)} ({time.time()-t0:.1f}s)")


In [ ]:
# =============================================================================
# ENTRAÎNEMENT DU MÉTA-MODÈLE (XGBoost sur les probabilités OOF)
# =============================================================================
t0 = time.time()

# Vote de direction (baseline de comparaison)
def direction_vote(meta_X_test_df, trained_ids, model_metrics_dict, threshold=0.5):
    """Vote pondéré par F1_dir de chaque modèle."""
    scores_up   = np.zeros(len(meta_X_test_df))
    scores_down = np.zeros(len(meta_X_test_df))
    for mid in trained_ids:
        f1 = model_metrics_dict.get(mid,{}).get('F1_dir', 0.5)
        if f1 < 0.50: continue
        p_up   = meta_X_test_df[[f'{mid}_c{k}' for k in [2,3] if f'{mid}_c{k}' in meta_X_test_df.columns]].sum(axis=1)
        p_down = meta_X_test_df[[f'{mid}_c{k}' for k in [0,1] if f'{mid}_c{k}' in meta_X_test_df.columns]].sum(axis=1)
        scores_up   += f1 * p_up.values
        scores_down += f1 * p_down.values
    return (scores_up > scores_down).astype(int)

# Vote baseline
y_vote = direction_vote(meta_X_test, meta_model_ids, model_metrics)
vote_f1 = f1_score((y_meta_dir_test==1).astype(int), y_vote, average='macro', zero_division=0)
print(f"Vote pondéré F1_dir = {vote_f1:.4f}")

# Méta-modèle XGBoost — cible binaire UP/DOWN
print("\nEntraînement méta-modèle XGBoost (direction binaire)...")
meta_clf_dir = XGBClassifier(
    n_estimators=CONFIG['meta_n_estimators'],
    max_depth=CONFIG['meta_max_depth'],
    learning_rate=CONFIG['meta_lr'],
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=SEED, n_jobs=-1
)
meta_clf_dir.fit(meta_X_train.values, y_meta_dir_train)
y_pred_meta_dir = meta_clf_dir.predict(meta_X_test.values)
y_prob_meta_dir = meta_clf_dir.predict_proba(meta_X_test.values)[:,1]

meta_dir_f1  = f1_score(y_meta_dir_test, y_pred_meta_dir, average='macro', zero_division=0)
meta_dir_acc = accuracy_score(y_meta_dir_test, y_pred_meta_dir)
print(f"  Méta-modèle direction F1_dir={meta_dir_f1:.4f} Acc={meta_dir_acc:.4f}")

# Méta-modèle XGBoost — cible 4 classes amplitude
print("\nEntraînement méta-modèle XGBoost (amplitude 4 classes)...")
meta_clf_4cls = XGBClassifier(
    n_estimators=CONFIG['meta_n_estimators'],
    max_depth=CONFIG['meta_max_depth'],
    learning_rate=CONFIG['meta_lr'],
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='mlogloss', objective='multi:softprob',
    random_state=SEED, n_jobs=-1
)
meta_clf_4cls.fit(meta_X_train.values, y_meta_4cls_train)
y_pred_meta_4cls = meta_clf_4cls.predict(meta_X_test.values)
meta_4cls_met = compute_metrics(y_meta_4cls_test, y_pred_meta_4cls)
print(f"  Méta-modèle 4cls F1_dir={meta_4cls_met['F1_dir']:.4f} F1_UP_FORT={meta_4cls_met.get('F1_UP_FORT',0):.4f}")

print(f"\n[DONE] Stacking terminé en {time.time()-t0:.1f}s")


In [ ]:
# =============================================================================
# RAPPORT FINAL
# =============================================================================
ML_REFS = {
    'LogReg N=9 (h=5j GLOBAL ref)':  {'F1_dir':0.634,'F1_UP_FORT':0.406,'F1_DOWN_FORT':0.575},
    'RandomForest N=8 (h=5j GLOBAL)':{'F1_dir':0.620,'F1_UP_FORT':0.412,'F1_DOWN_FORT':0.462},
}

print("="*70)
print("RAPPORT FINAL — Modèles individuels par horizon")
print("="*70)
for h in CONFIG['horizons']:
    models_h = {mid: met for mid,met in model_metrics.items() if met.get('horizon')==h}
    if not models_h: continue
    best = max(models_h.items(), key=lambda x: x[1].get('F1_dir',0))
    print(f"\nh={h}j ({len(models_h)} modèles) | Best: {best[0]} F1_dir={best[1].get('F1_dir',0):.4f}")
    for mid,met in sorted(models_h.items(), key=lambda x: -x[1].get('F1_dir',0)):
        print(f"  {mid:<55} F1_dir={met.get('F1_dir',0):.4f} "
              f"UP_FORT={met.get('F1_UP_FORT',0):.4f} DOWN_FORT={met.get('F1_DOWN_FORT',0):.4f}")

print("\n" + "="*70)
print("RAPPORT STACKING — Méta-modèle multi-horizon")
print("="*70)
print(f"  Vote pondéré F1_dir          = {vote_f1:.4f}")
print(f"  Méta XGBoost (direction)     = {meta_dir_f1:.4f}")
print(f"  Méta XGBoost (4 classes)     F1_dir={meta_4cls_met['F1_dir']:.4f} "
      f"UP_FORT={meta_4cls_met.get('F1_UP_FORT',0):.4f} "
      f"DOWN_FORT={meta_4cls_met.get('F1_DOWN_FORT',0):.4f}")
print("─"*70)
for k,v in ML_REFS.items():
    print(f"  {k:<50} F1_dir={v['F1_dir']:.4f}")

# SHAP sur le méta-modèle
print("\n[SHAP] Feature importance du méta-modèle...")
try:
    explainer = shap.TreeExplainer(meta_clf_4cls)
    sv = explainer.shap_values(meta_X_test.values[:300])
    if isinstance(sv, list): arr = np.mean([np.abs(s) for s in sv], axis=0)
    elif np.array(sv).ndim==3: arr = np.abs(sv).mean(axis=2)
    else: arr = np.abs(sv)
    shap_scores = pd.Series(arr.mean(axis=0), index=meta_X_test.columns)
    top_shap = shap_scores.nlargest(20)
    print("  Top-20 features du méta-modèle (SHAP) :")
    for feat, score in top_shap.items():
        model_src = feat.split('_c')[0] if '_c' in feat else feat
        print(f"    {feat:<60} {score:.5f}")
except Exception as e:
    print(f"  [WARN] SHAP: {e}")

# Export Excel
try:
    rows_indiv = []
    for mid, met in model_metrics.items():
        rows_indiv.append({'Model_ID':mid, **met})
    rows_meta = [
        {'Model_ID':'Vote_Pondéré','F1_dir':vote_f1,'type':'ensemble'},
        {'Model_ID':'Meta_XGB_dir','F1_dir':meta_dir_f1,'type':'stacking'},
        {'Model_ID':'Meta_XGB_4cls',**meta_4cls_met,'type':'stacking'},
    ]
    with pd.ExcelWriter('vix_multihorizon_stacking_report.xlsx', engine='xlsxwriter') as w:
        pd.DataFrame(rows_indiv).sort_values('F1_dir',ascending=False).to_excel(w,'Individual_Models',index=False)
        pd.DataFrame(rows_meta).to_excel(w,'Stacking_Results',index=False)
        top_shap.reset_index().rename(columns={'index':'Feature',0:'SHAP'}).to_excel(w,'SHAP_Meta',index=False)
    print("\n[SAVE] vix_multihorizon_stacking_report.xlsx")
except Exception as e:
    print(f"[WARN] Export: {e}")

print("\n[NOTE] Aucun modèle enregistré — validation explicite requise.")
